In [14]:
from __future__ import annotations

import sys
from pathlib import Path

from src.infrastructure import RuntimeContext
from src.services import CanonicalPlanningService

ROOT = Path(".").resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.services.query_understanding import QueryUnderstandingService
from rich import print as rprint
from src.application.contracts import PipelineRequest
from src.application.settings import ViRAGESettings
from src.application import ViRAGEPipeline

In [15]:
from dotenv import load_dotenv
load_dotenv("../.env")

True

In [16]:
tmp_path = Path("../examples/tmp_folder")
data_path = Path("../examples/demo.csv")
settings = ViRAGESettings(artifact_root=tmp_path / "artifacts")
pipeline = ViRAGEPipeline(settings)
request = PipelineRequest(query="Show the sales trend over time", data_path=data_path.as_posix())
result = pipeline.invoke(request)

In [17]:
print(result)
# rprint(result)

run_id='9f0446b765224ae780233971ed89a5ac' query='Show the sales trend over time' data_path='../examples/demo.csv' case_type=<ChartCaseType.CANONICAL: 'canonical'> query_understanding=QueryUnderstandingResult(intent='Show the sales trend over time', requested_operations=['trend analysis'], candidate_charts=['line'], constraints=[], case_type=<ChartCaseType.CANONICAL: 'canonical'>, confidence=0.8) planning=PlanningResult(mode=<ChartCaseType.CANONICAL: 'canonical'>, steps=[PlanningStep(name='profile_the_dataset_and_confirm_field', description='Profile the dataset and confirm field types relevant to the request.'), PlanningStep(name='prepare_a_cleaned_analysis_ready_version', description='Prepare a cleaned analysis-ready version of the data.'), PlanningStep(name='retrieve_concise_charting_guidance_for_the', description='Retrieve concise charting guidance for the selected chart family.'), PlanningStep(name='build_the_primary_requested_chart_using', description='Build the primary requested c

In [18]:
from langchain_ollama import ChatOllama
LLM_MODEL = "gemma3:1b"  # или "llama3.2:1b"
# LLM_MODEL = "llama3.2:1b"       # или "llama3.2:1b"
llm = ChatOllama(
    model=LLM_MODEL,
    temperature=0,
)

# QueryUnderstandingService

In [19]:
r = QueryUnderstandingService().invoke(user_context=request.user_context, runtime=RuntimeContext(settings=settings, llm=llm), query=request.query)
# r = QueryUnderstandingService().invoke(user_context=request.user_context, runtime=RuntimeContext(settings=settings), query=request.query)
rprint("query:",  request.query)
rprint(r)

query: Show the sales trend over time

QueryUnderstandingResult(
    intent='Show the sales trend over time',
    requested_operations=['trend analysis'],
    candidate_charts=['line'],
    constraints=[],
    case_type=<ChartCaseType.CANONICAL: 'canonical'>,
    confidence=0.8
)

In [20]:
# qu = QueryUnderstandingService().invoke(user_context=request.user_context, runtime=RuntimeContext(settings=settings, llm=llm), query=request.query)
qu = QueryUnderstandingService().invoke(user_context=request.user_context, runtime=runtime, query=request.query)
rprint("query:",  request.query)
rprint(qu)

query: Show the sales trend over time

QueryUnderstandingResult(
    intent='Show the sales trend over time',
    requested_operations=['trend analysis'],
    candidate_charts=['line'],
    constraints=[],
    case_type=<ChartCaseType.CANONICAL: 'canonical'>,
    confidence=0.8
)

In [27]:
cp = CanonicalPlanningService().invoke(query_understanding=qu, runtime=RuntimeContext(settings=settings, llm=llm))
# cp = CanonicalPlanningService().invoke(query_understanding=qu, runtime=RuntimeContext(settings=settings))

In [28]:
rprint(cp)

PlanningResult(
    mode=<ChartCaseType.CANONICAL: 'canonical'>,
    steps=[
        PlanningStep(
            name='profile_the_dataset_and_confirm_field',
            description='Profile the dataset and confirm field types relevant to the request.'
        ),
        PlanningStep(
            name='prepare_a_cleaned_analysis_ready_version',
            description='Prepare a cleaned analysis-ready version of the data.'
        ),
        PlanningStep(
            name='retrieve_concise_charting_guidance_for_the',
            description='Retrieve concise charting guidance for the selected chart family.'
        ),
        PlanningStep(
            name='build_the_primary_requested_chart_using',
            description='Build the primary requested chart using the leading chart family: line.'
        ),
        PlanningStep(
            name='execute_plotting_code_and_collect_numeric',
            description='Execute plotting code and collect numeric summaries from the run.'
        ),
        PlanningStep(
            name='read_chart_structure,_extract_facts_and',
            description='Read chart structure, extract facts and verify that final statements are evidence-backed.'
        )
    ],
    success_criteria=[
        'At least one valid canonical chart is produced, preferably among: line.',
        'Generated charts are readable and consistent with the request.',
        'Final statements reference execution metrics or chart evidence.'
    ]
)